# Private T4 compatibility smoke
Five accepted training questions, alternating EN/VI, ten unchanged native CLI17 processes. No benchmark evaluation or BigQuery. Cache input read-only; full pinned runtime installed separately and removed after execution. Stop at first runtime failure. 300s per process, 1800s smoke, 2700s session. Research BF16/4-bit configuration unchanged. No HF secret required.


In [ ]:
ALLOW_GPU_SMOKE = False
CAMPAIGN = "t4-smoke-20261017-r01"
BUNDLE_SHA256 = "2aa4dac0cb4110783279c3361b4c7618ce601402ed3a40cef168adad75caa6de"
EXPECTED_SETUP_FILES = {'scripts/26_kaggle_environment.py': '91be7869ae7223b93e7ff6c3c7a58589b241161ef7a782351f02e105e86d9218', 'configs/kaggle-b12-model-pins.json': '6f72c03fe03c379b1ed0844192ef17c94cb60e908c5269f0e49750a32f04998a', 'configs/kaggle-b12-requirements.lock.txt': 'da3c09c04886fca688aa6a2e54aa1d7c46726be5a4c67b19bc2ba8b9ab36a5cd', 'configs/kaggle-b12-bootstrap.lock.txt': '021d7fd19573db49d3acd74056047912457790f20ceffb4d1196ebffa067a9c8'}
WORKER_SHA256 = 'db94a4196d9169fa2fe9d0e0d7e47e0db45629c3392abba5537172b04db595d5'
WORKER_SOURCE = '"""Operational non-test T4 probe using unchanged native CLI17; never evaluate test data."""\nimport argparse\nimport hashlib\nimport importlib.metadata\nimport json\nimport statistics\nimport subprocess\nimport sys\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nTRAINING_SHA = \'b6440c0b281a270907d8dd5b8d11cdd1506ce125ae465aa63ee66963e0dd4835\'\nMODEL_REVISION = \'8afb486c1db24fe5011ec46dfbe5b5dccdb575c2\'\nENCODER_REVISION = \'1110a243fdf4706b3f48f1d95db1a4f5529b4d41\'\nNATIVE_CAP_SECONDS = 300\nSMOKE_CAP_SECONDS = 1800\n\n\ndef utc():\n    return datetime.now(timezone.utc).isoformat()\n\n\ndef write_json(path, payload):\n    if path.exists():\n        raise ValueError(\'Refuse to overwrite existing smoke evidence: \' + str(path))\n    path.write_text(json.dumps(payload, ensure_ascii=False, sort_keys=True, indent=2) + \'\\n\')\n\n\ndef select_cases(training_bytes):\n    if hashlib.sha256(training_bytes).hexdigest() != TRAINING_SHA:\n        raise ValueError(\'Accepted training fingerprint mismatch\')\n    rows = [json.loads(line) for line in training_bytes.decode().splitlines()]\n    selected = []\n    families = set()\n    for language in (\'en\', \'vi\', \'en\', \'vi\', \'en\'):\n        row = next(r for r in rows if r[\'split\'] == \'train\' and r[\'language\'] == language and r[\'semantic_family_id\'] not in families)\n        selected.append({key: row[key] for key in (\'id\', \'language\', \'semantic_family_id\', \'question\', \'record_sha256\')})\n        families.add(row[\'semantic_family_id\'])\n    return selected\n\n\ndef as_text(value):\n    return value.decode(\'utf-8\', errors=\'replace\') if isinstance(value, bytes) else (value or \'\')\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--root\', type=Path, required=True)\n    parser.add_argument(\'--hf-home\', type=Path, required=True)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    parser.add_argument(\'--campaign\', required=True)\n    args = parser.parse_args()\n    root, output = args.root, args.output\n    output.mkdir(parents=True, exist_ok=False)\n    started = utc()\n    summary = {\'mode\': \'non_test_train_smoke\', \'campaign_id\': args.campaign, \'started_at\': started,\n               \'status\': \'blocked\', \'scientific_ready\': False, \'benchmark_evaluation\': False,\n               \'training_sha256\': TRAINING_SHA, \'model_revision\': MODEL_REVISION,\n               \'encoder_revision\': ENCODER_REVISION, \'native_cap_seconds\': NATIVE_CAP_SECONDS,\n               \'smoke_cap_seconds\': SMOKE_CAP_SECONDS, \'cases\': [], \'processes\': []}\n    try:\n        from nl2sparql.kaggle.bundle import preflight_data\n        from nl2sparql.kaggle.runner import offline_environment, preflight_runtime, probe_hardware\n        from nl2sparql.models.b12.retrieval import validate_training_snapshot\n        training = root / \'data/dataset/processed/b2-training.jsonl\'\n        data = training.read_bytes()\n        records, digest = validate_training_snapshot(data, accepted_training_sha256=TRAINING_SHA)\n        if len(records) != 7208 or digest != TRAINING_SHA:\n            raise ValueError(\'Training authority mismatch\')\n        cases = select_cases(data)\n        summary[\'cases\'] = cases\n        write_json(output / \'smoke-cases.json\', {\'selection\': \'First matching rows in accepted train order, alternating EN/VI, distinct families, fixed before generation\', \'cases\': cases, \'training_sha256\': TRAINING_SHA})\n        config = json.loads((root / \'configs/kaggle-b12-run.template.json\').read_bytes())\n        config.update(campaign_id=args.campaign, language=\'en\', model_revision=MODEL_REVISION,\n                      encoder_revision=ENCODER_REVISION, hf_home=str(args.hf_home), run_timeout_seconds=NATIVE_CAP_SECONDS)\n        write_json(output / \'runtime-config.json\', config)\n        data_check = preflight_data(root)\n        write_json(output / \'data-preflight.json\', data_check)\n        if data_check[\'status\'] != \'data_ready\':\n            raise ValueError(\'Data preflight blocked\')\n        runtime_check = preflight_runtime(root, config, output)\n        write_json(output / \'runtime-preflight.json\', runtime_check)\n        if runtime_check[\'status\'] != \'ready\':\n            raise ValueError(\'Runtime preflight blocked: \' + \'; \'.join(runtime_check[\'blockers\']))\n        write_json(output / \'packages.json\', sorted((d.metadata[\'Name\'], d.version) for d in importlib.metadata.distributions()))\n        hardware = probe_hardware()\n        write_json(output / \'hardware.json\', hardware)\n        print(\'Actual hardware:\', hardware, flush=True)\n        child_env = offline_environment(root, config)\n        begin = time.monotonic()\n        for baseline in (\'b1\', \'b2\'):\n            for index, case in enumerate(cases, 1):\n                remaining = SMOKE_CAP_SECONDS - (time.monotonic() - begin)\n                if remaining <= 0:\n                    raise TimeoutError(\'Smoke wall-time budget exhausted\')\n                folder = output / f\'{baseline}-case-{index:02d}\'\n                folder.mkdir()\n                argv = [sys.executable, str(root / \'scripts/17_small_llm_baselines.py\'), \'predict\',\n                        \'--baseline\', baseline, \'--question\', case[\'question\'], \'--model-revision\', MODEL_REVISION,\n                        \'--catalog\', str(root / \'src/nl2sparql/sql/catalog/ethereum_analytics.json\'),\n                        \'--training\', str(training), \'--cache\', str(output / \'cache/b2-few-shot.npz\'),\n                        \'--encoder-id\', \'sentence-transformers/all-MiniLM-L6-v2\', \'--encoder-revision\', ENCODER_REVISION,\n                        \'--accepted-training-sha256\', TRAINING_SHA, \'--target-id\', case[\'id\'], \'--real-inference\']\n                record = {\'baseline\': baseline, \'case_id\': case[\'id\'], \'language\': case[\'language\'],\n                          \'argv\': argv, \'started_at\': utc(), \'timeout_seconds\': min(NATIVE_CAP_SECONDS, remaining)}\n                print(\'Starting\', baseline, case[\'id\'], flush=True)\n                tick = time.monotonic()\n                try:\n                    process = subprocess.run(argv, env=child_env, text=True, capture_output=True, timeout=record[\'timeout_seconds\'])\n                    stdout, stderr, code = process.stdout, process.stderr, process.returncode\n                except subprocess.TimeoutExpired as exc:\n                    stdout, stderr, code = as_text(exc.stdout), as_text(exc.stderr), 124\n                    record[\'timed_out\'] = True\n                (folder / \'native-stdout.json\').write_text(stdout)\n                (folder / \'native-stderr.log\').write_text(stderr)\n                record.update(exit_code=code, finished_at=utc(), wall_seconds=time.monotonic() - tick)\n                try:\n                    payload = json.loads(stdout)\n                except (ValueError, TypeError):\n                    payload = {\'status\': \'unparseable_native_stdout\'}\n                record[\'native_status\'] = payload.get(\'status\')\n                if payload.get(\'status\') == \'ready\':\n                    prediction = payload[\'prediction\']\n                    if prediction[\'completion\'][\'synthetic_backend\'] is not False:\n                        raise ValueError(\'Non-genuine completion marker in smoke\')\n                    record.update(latency_ms=prediction[\'latency_ms\'], extraction_status=prediction[\'extraction_status\'],\n                                  model_revision=prediction[\'completion\'][\'model_revision\'],\n                                  synthetic_backend=prediction[\'completion\'][\'synthetic_backend\'])\n                else:\n                    record[\'error\'] = payload.get(\'error\', payload.get(\'status\'))\n                write_json(folder / \'process.json\', record)\n                summary[\'processes\'].append(record)\n                print(\'Finished\', baseline, case[\'id\'], \'exit\', code, \'native status\', payload.get(\'status\'), flush=True)\n                if code != 0 or payload.get(\'status\') != \'ready\':\n                    raise RuntimeError(str(record.get(\'error\', \'Native prediction failed\')))\n        summary[\'status\'] = \'passed\'\n        summary[\'latency_observations\'] = {\n            baseline: {\'cases\': 5, \'median_ms\': statistics.median(r[\'latency_ms\'] for r in summary[\'processes\'] if r[\'baseline\'] == baseline),\n                       \'max_ms\': max(r[\'latency_ms\'] for r in summary[\'processes\'] if r[\'baseline\'] == baseline)}\n            for baseline in (\'b1\', \'b2\')}\n    except Exception as exc:\n        summary[\'error_type\'] = type(exc).__name__\n        summary[\'error\'] = str(exc)\n        print(\'Smoke blocked:\', type(exc).__name__, str(exc), flush=True)\n    finally:\n        summary[\'finished_at\'] = utc()\n        write_json(output / \'smoke-summary.json\', summary)\n        files = {str(p.relative_to(output)): {\'bytes\': p.stat().st_size, \'sha256\': hashlib.sha256(p.read_bytes()).hexdigest()}\n                 for p in output.rglob(\'*\') if p.is_file()}\n        write_json(output / \'smoke-artifacts-manifest.json\', files)\n    return 0 if summary[\'status\'] == \'passed\' else 2\n\n\nif __name__ == \'__main__\':\n    raise SystemExit(main())\n'


In [ ]:
import hashlib, json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path, PurePosixPath
if not ALLOW_GPU_SMOKE:
    print("BLOCKED: GPU smoke disabled")
else:
    stage = "input_discovery"
    started = time.monotonic()
    working = Path("/kaggle/working")
    evidence = working / "b12-smoke-driver"
    evidence.mkdir(exist_ok=False)
    workspace = working / "b12-input"
    runtime = working / "b12-runtime"
    bootstrap = working / "b12-bootstrap"
    uv_cache = working / "b12-uv-cache"
    managed = working / "b12-python"
    for folder in (workspace, runtime, bootstrap, uv_cache, managed):
        if folder.exists():
            raise ValueError("Fresh session required")
    env = {k: os.environ[k] for k in ("PATH", "LD_LIBRARY_PATH", "CUDA_VISIBLE_DEVICES", "LANG", "LC_ALL") if k in os.environ}
    env.update(UV_CACHE_DIR=str(uv_cache), UV_PYTHON_INSTALL_DIR=str(managed), HF_HUB_DISABLE_IMPLICIT_TOKEN="1", HF_HUB_DISABLE_TELEMETRY="1")
    result = {"status": "blocked", "stage": stage, "campaign_id": CAMPAIGN, "session_cap_seconds": 2700, "smoke_cap_seconds": 1800}
    def run(argv, *, custom_env=None, output_name=None):
        print("Stage:", stage, flush=True)
        remaining = 2580 - (time.monotonic() - started)
        if remaining <= 0:
            raise TimeoutError("Session budget exhausted")
        with (evidence / (output_name or (stage + ".log"))).open("w") as log:
            proc = subprocess.run(argv, env=custom_env or env, stdout=log, stderr=subprocess.STDOUT, text=True, timeout=remaining)
        if proc.returncode:
            raise RuntimeError("Stage failed: " + stage + " exit=" + str(proc.returncode))
    try:
        archives = list(Path("/kaggle/input").rglob("input-bundle.bundle"))
        inventories = list(Path("/kaggle/input").rglob("b12-cache-inventory.json"))
        if len(archives) != 1 or len(inventories) != 1:
            raise ValueError("Exactly one accepted bundle and cache inventory required")
        archive, inventory = archives[0], inventories[0]
        kit = archive.parent
        hf_home = inventory.parent / "hf-home"
        if not hf_home.is_dir():
            raise ValueError("Mounted cache missing")
        for name, digest in EXPECTED_SETUP_FILES.items():
            if hashlib.sha256((kit/name).read_bytes()).hexdigest() != digest:
                raise ValueError("Setup fingerprint mismatch: " + name)
        if hashlib.sha256(archive.read_bytes()).hexdigest() != BUNDLE_SHA256:
            raise ValueError("Accepted bundle fingerprint mismatch")
        with tarfile.open(archive) as tar:
            entries = tar.getmembers()
            if len(entries) != 181 or len({m.name for m in entries}) != len(entries) or sum(m.size for m in entries) > 128*1024*1024:
                raise ValueError("Bundle member count/size invalid")
            for item in entries:
                path = PurePosixPath(item.name)
                if not item.isfile() or path.is_absolute() or any(p in {".", ".."} or p.startswith(".") for p in item.name.split("/")):
                    raise ValueError("Unsafe bundle entry")
            tar.extractall(workspace, filter="data")
        (evidence/"input-provenance.json").write_text(json.dumps({"bundle_sha256": BUNDLE_SHA256, "setup_files": EXPECTED_SETUP_FILES, "cache_inventory_sha256": hashlib.sha256(inventory.read_bytes()).hexdigest(), "cache_inventory": json.loads(inventory.read_bytes()), "hf_home": str(hf_home)}, indent=2))
        stage = "install_bootstrap"
        run([sys.executable, "-m", "pip", "--isolated", "install", "--no-deps", "--require-hashes", "--target", str(bootstrap), "-r", str(kit/"configs/kaggle-b12-bootstrap.lock.txt")])
        uv = str(bootstrap/"bin/uv")
        stage = "create_python311"
        run([uv, "--no-config", "venv", "--python", "3.11.15", "--managed-python", str(runtime)])
        stage = "install_locked_environment"
        run([uv, "--no-config", "--no-cache", "pip", "sync", "--python", str(runtime/"bin/python"), "--require-hashes", "--index-url", "https://pypi.org/simple", str(kit/"configs/kaggle-b12-requirements.lock.txt")])
        offline = dict(env, PYTHONPATH=str(workspace/"src"), PYTHONDONTWRITEBYTECODE="1", HF_HOME=str(hf_home), HF_HUB_CACHE=str(hf_home/"hub"), HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TOKENIZERS_PARALLELISM="false")
        stage = "verify_mounted_cache"
        run([str(runtime/"bin/python"), str(kit/"scripts/26_kaggle_environment.py"), "verify-cache", "--pins", str(kit/"configs/kaggle-b12-model-pins.json"), "--hf-home", str(hf_home), "--component", "all"], custom_env=offline, output_name="verified-mounted-cache.json")
        worker_path = evidence/"smoke_worker.py"
        if hashlib.sha256(WORKER_SOURCE.encode()).hexdigest() != WORKER_SHA256:
            raise ValueError("Operational worker fingerprint mismatch")
        worker_path.write_text(WORKER_SOURCE)
        stage = "native_cli17_smoke"
        run([str(runtime/"bin/python"), str(worker_path), "--root", str(workspace), "--hf-home", str(hf_home), "--output", str(working/"b12-smoke-results"), "--campaign", CAMPAIGN], custom_env=offline)
        result["status"] = "passed"
    except Exception as exc:
        result.update(error_type=type(exc).__name__, error=str(exc))
        print("Blocked:", stage, type(exc).__name__, str(exc), flush=True)
    finally:
        result.update(stage=stage, wall_seconds=time.monotonic()-started)
        for folder in (workspace, runtime, bootstrap, uv_cache, managed):
            if folder.exists():
                shutil.rmtree(folder)
        (evidence/"driver-summary.json").write_text(json.dumps(result, indent=2))
        print("Driver result:", result, flush=True)
    if result["status"] != "passed":
        raise RuntimeError("Smoke blocked; inspect retained evidence")
